In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

True

## Setup the configuration

In [38]:
class Config:
    # setup mistral configuration
    mistral_api_key = os.getenv("MISTRAL_API_KEY")
    mistral_chat_model = os.getenv("MISTRAL_CHAT_MODEL")
    mistral_embed_model = "mistral-embed"
    mistral_embed_dimension =  int(os.getenv("MISTRAL_EMBED_DIMENSION")) or 1024

    # pinecone configuration
    pinecone_api_key = os.getenv("PINECONE_API_KEY")
    pinecone_index = "rag-system"
    pinecone_namespace = os.getenv("PINECONE_NAMESPACE")
    pinecone_region = os.getenv("PINECONE_REGION")
    pinecone_reranker_model = "bge-reranker-v2-m3"
    pinecone_retriever_count = 40
    pinecone_reranker_count = 10

    # document path details
    document_dir = "docs"

    # text splitter
    chunk_size = 1000
    overlap_size = 200

## Setup the mistral service

In [3]:
from langchain_mistralai import ChatMistralAI, MistralAIEmbeddings

class MistralService:
    chatModel : ChatMistralAI
    embeddingModel : MistralAIEmbeddings

    def __init__(self):
        self.chatModel = self.connectMistralChatModel()
        self.embeddingModel = self.connectMistralEmbedModel()

    def connectMistralChatModel(self, model_name : str = Config.mistral_chat_model) -> ChatMistralAI :
        try:
            return ChatMistralAI(
                api_key = Config.mistral_api_key,
                model = model_name
            )
        except Exception as e:
            print(f"Something went wrong in the {model_name} connection...")
            raise
        
    def connectMistralEmbedModel(self, model_name: str = Config.mistral_embed_model) -> MistralAIEmbeddings :
        try:
            return MistralAIEmbeddings(
                api_key = Config.mistral_api_key,
                model = model_name
            )
        except Exception as e:
            print(f"Something went wrong in the {model_name} connection...")
            raise
    
    def getChatModel(self) -> ChatMistralAI:
        return self.chatModel
    
    def getEmbedModel(self) -> MistralAIEmbeddings:
        return self.embeddingModel

## Setup the Pinecone service

In [4]:
from pinecone import Pinecone, ServerlessSpec
from langchain_pinecone import PineconeVectorStore

class PineconeService:
    def __init__(self):
        self.pc = Pinecone(api_key=Config.pinecone_api_key) 
        self.prepare_pinecone()

    def prepare_pinecone(self):
        index_list_obj = self.pc.list_indexes()
        index_names = [index.name for index in index_list_obj]

        spec = ServerlessSpec(
            cloud="aws",
            region=Config.pinecone_region
        )

        if Config.pinecone_index not in index_names:
            try:
                print(f"Creating index: {Config.pinecone_index}")
                self.pc.create_index(
                    name=Config.pinecone_index,
                    dimension=Config.mistral_embed_dimension,
                    metric="cosine",
                    spec=spec
                )
            except Exception as e:
                print(f"Error during index creation: {e}")
                raise
        else:
            print(f"Index '{Config.pinecone_index}' already exists. Skipping creation.")

    def getPinecone(self) -> Pinecone:
        return self.pc

    def getPineconeStore(self, embedding) -> PineconeVectorStore:
        self.vectorStore = PineconeVectorStore(
            index_name= Config.pinecone_index,
            embedding= embedding
        )

        return self.vectorStore

/home/buddhika-madusanka/projects/industrial-rag-system/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Setup retriever

In [39]:
query = "What is rag system is ?"

pc = PineconeService().getPineconeStore(MistralService().getEmbedModel())
reuslts = pc.similarity_search(query, k= Config.pinecone_retriever_count)

raw_rag_result_content = "".join([res.page_content for res in reuslts])

Index 'rag-system' already exists. Skipping creation.


## Setup the reranker

In [40]:
from langchain_pinecone import PineconeRerank

reranker = PineconeRerank(model= Config.pinecone_reranker_model, top_n= Config.pinecone_reranker_count)

rerank_docs = reranker.compress_documents(
    reuslts, query
)

rerank_document_content = "".join([res.page_content for res in rerank_docs])

## Check the quality of the normal retriever results and retriever results

In [41]:
from langchain_core.prompts import PromptTemplate

prompt_message = """
    Role:
    You are a Content Quality Auditor specialized in Data post-processing for RAG systems.
    
    Task:
    Your task is the generate the value how much that document contents quality is ?. Evaluate the "Informational Density" and the relvense of the provided content with the query use provided. Your goal is to distinguish between high-value knowledge and low-value structural noise.

    statement:
    {statement}

    query:
    {query}
"""

prompt_template = PromptTemplate.from_template(prompt_message)

def qualityChecker(content: str, query: str) -> PromptTemplate:
    return prompt_template.invoke({
        "statement" : content,
        "query" : query
    })

In [42]:
from pydantic import BaseModel, Field

class QualityCheckerOuput(BaseModel):
    quality_result : float = Field(..., ge=0 , le= 1)
    statement: str = Field(..., description= "What is the reason for the quality result")

llm = MistralService().getChatModel()
structured_llm = llm.with_structured_output(QualityCheckerOuput)

In [45]:
raw_content_quality_prompt = qualityChecker(raw_rag_result_content, query)
reranker_content_prompt = qualityChecker(rerank_document_content, query)

raw_content_result = structured_llm.invoke(raw_content_quality_prompt)
rerank_content_result = structured_llm.invoke(reranker_content_prompt)

# Results comparison

In [46]:
print("=" * 10 + "Raw content" + "=" * 10)
print(f"Raw content awareness quality with the query : {raw_content_result.quality_result}")
print(f"Raw content awareness quality reason : {raw_content_result.statement}")

print("=" * 10 + "Reranker content" + "=" * 10)
print(f"Raw content awareness quality with the query : {rerank_content_result.quality_result}")
print(f"Raw content awareness quality reason : {rerank_content_result.statement}")

==========Raw content==========
Raw content awareness quality with the query : 0.7
Raw content awareness quality reason : The provided content segments related to the query 'What is RAG system?' are highly relevant but are scattered across different parts of the document. While the context discusses Retrieval-Augmented Generation (RAG) in detail, the direct explanation of what RAG is is not explicitly summarized or highlighted in a concise manner within the given text. The document contains extensive technical details about RAG's architecture, performance metrics, experiments, and comparisons with other models, but the core definition or foundational explanation of RAG is implied rather than explicitly stated in the response sections. To fully address the query, a more direct and focused summary of the RAG system's purpose and basic principles would improve informational density.
==========Reranker content==========
Raw content awareness quality with the query : 0.92
Raw content awaren